<a href="https://colab.research.google.com/github/sejal-godbole/Federated-Learning/blob/main/FL_Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler

In [2]:
np.random.seed(42)

NUM_SAMPLES = 8000

area = np.random.uniform(500, 3000, NUM_SAMPLES)
bedrooms = np.random.randint(1, 6, NUM_SAMPLES)
age = np.random.uniform(0, 30, NUM_SAMPLES)

price = (
    50 * area +
    100000 * bedrooms -
    2000 * age +
    np.random.normal(0, 20000, NUM_SAMPLES)
)

data = pd.DataFrame({
    "area": area,
    "bedrooms": bedrooms,
    "age": age,
    "price": price
})

In [3]:
X = data.drop("price", axis=1)
y = data["price"]

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [4]:
NUM_CLIENTS = 4

full_data = pd.concat([X_scaled, y.reset_index(drop=True)], axis=1)
full_data = full_data.sample(frac=1, random_state=1).reset_index(drop=True)

client_data = np.array_split(full_data, NUM_CLIENTS)

clients = {}
for i in range(NUM_CLIENTS):
    clients[f"Client_{i+1}"] = client_data[i]

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [5]:
def train_local_model(client_df):
    X_client = client_df.drop("price", axis=1)
    y_client = client_df["price"]

    model = LinearRegression()
    model.fit(X_client, y_client)

    return model.coef_, model.intercept_, len(client_df)


In [6]:
def federated_averaging(client_updates):
    total_samples = sum(n for _, _, n in client_updates)

    avg_weights = np.zeros_like(client_updates[0][0])
    avg_bias = 0

    for weights, bias, n in client_updates:
        avg_weights += (n / total_samples) * weights
        avg_bias += (n / total_samples) * bias

    return avg_weights, avg_bias


In [7]:
ROUNDS = 5

global_weights = None
global_bias = 0

for round_num in range(ROUNDS):
    print(f"\n--- Federated Round {round_num + 1} ---")

    client_updates = []

    for client_name, client_df in clients.items():
        weights, bias, samples = train_local_model(client_df)
        client_updates.append((weights, bias, samples))
        print(f"{client_name} trained on {samples} samples")

    global_weights, global_bias = federated_averaging(client_updates)

print("\nGlobal Model Weights:", global_weights)
print("Global Model Bias:", global_bias)



--- Federated Round 1 ---
Client_1 trained on 2000 samples
Client_2 trained on 2000 samples
Client_3 trained on 2000 samples
Client_4 trained on 2000 samples

--- Federated Round 2 ---
Client_1 trained on 2000 samples
Client_2 trained on 2000 samples
Client_3 trained on 2000 samples
Client_4 trained on 2000 samples

--- Federated Round 3 ---
Client_1 trained on 2000 samples
Client_2 trained on 2000 samples
Client_3 trained on 2000 samples
Client_4 trained on 2000 samples

--- Federated Round 4 ---
Client_1 trained on 2000 samples
Client_2 trained on 2000 samples
Client_3 trained on 2000 samples
Client_4 trained on 2000 samples

--- Federated Round 5 ---
Client_1 trained on 2000 samples
Client_2 trained on 2000 samples
Client_3 trained on 2000 samples
Client_4 trained on 2000 samples

Global Model Weights: [123284.79485251 400344.21651286 -60512.58348754]
Global Model Bias: 125761.922270654
